In [ ]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [ ]:
"""Skip during test
series_folder = Path("/path/to/section-set")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

In [ ]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [ ]:
series_folder = Path("/path/to/section-set")
# The axial spacing remains a user-provided value in microns.
from atlas.io.fibics_metadata import get_stitched_pixel_size_microns

stitched_tifs = sorted(series_folder.glob("stitched_image_S_*.tiff"))
if not stitched_tifs:
    raise FileNotFoundError(f"No stitched section TIFFs found in {series_folder}")

with tiff.TiffFile(stitched_tifs[0]) as tif:
    acquisition_metadata = tif.shaped_metadata[0]["AcquisitionMetadata"]
pix_size_micron = get_stitched_pixel_size_microns(stitched_tifs[0])
print(f"Using MIF pixel size: {pix_size_micron} µm from {stitched_tifs[0].name}")
for axis in ("X", "Y"):
    value = acquisition_metadata.get(f"Pixel size tile TIFF {axis}")
    unit = acquisition_metadata.get(f"Pixel size tile TIFF {axis} unit")
    if value is not None:
        print(f"Tile TIFF pixel size {axis}: {value} {unit}")

pixel_size = {"Value": pix_size_micron, "Axial": 0.300, "Unit": "µm"}

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

In [ ]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)
from atlas.io import reorder_files_by_s_number, parse_shorthand_order

In [ ]:
correct_order_path = Path(series_folder.joinpath("correct_order.txt"))

try:
    new_order = parse_shorthand_order(correct_order_path)
    print(f"Loaded correct_order.txt: {new_order}")

    tif_list_sorted = reorder_files_by_s_number(tif_list_sorted, new_order)
    print("Files reordered based on correct_order.txt:")
    for f in tif_list_sorted:
        print(f)

except FileNotFoundError:
    print("No correct_order.txt found — using original order.")
except Exception as e:
    print(f"Error reading or applying correct_order.txt: {e}")

In [ ]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif
from atlas.io import create_empty_folder, rm_tree, apply_alignment, zarr_array_to_czi
from atlas.image_analysis import image_dtype_min_max, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment, first_last_true, ROI, correct_z_alignment_from_points
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity


In [ ]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

#TODO: implement handling of outliers in the alignment results, this can be done by looking at the current shift and then decide whichones looks weird.

In [ ]:
z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

In [ ]:
zarr.__version__

In [ ]:
# now we save a downsample version of the alignment for visual inspection
use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, percentile_low=2.0, percentile_high=99.95, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
# this is an example of how to apply a manual alignment correction, please ignore if not needed, 
# Define corresponding landmarks after inspecting the initially aligned images.
# Each point uses (z, y, x): the fixed point belongs to the earlier section and
# the matching moving point belongs to the immediately following section. For
# example, [1, 692, 391] and [2, 660, 376] identify the same feature in two
# consecutive sections. Coordinates must use the same pixel scale as the shifts, meaning inspect in the DS data.
# Additional pairs can be appended in matching order; points from the same section
# pair are combined using their median residual correction.
do_correction = False
if do_correction:
    z_align_df = pd.read_pickle(z_align_df_path)
    # index should be given in z, y, x
    list_of_fix_points = [np.array([1, 692., 391.])]
    list_of_mov_points = [np.array([2, 660., 376.])]

    z_align_df = correct_z_alignment_from_points(
        z_align_df,
        list_of_fix_points,
        list_of_mov_points,
    )

    use_down_sample = True
    zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, percentile_low=2.0, percentile_high=99.95, use_down_sample=use_down_sample)

    out_pixel_size = pixel_size.copy()

    if use_down_sample:
        out_pixel_size['Value'] = pixel_size['Value'] * down_scale
        end_str = "_ds_aligned"
    else:
        end_str = "_aligned"

    czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [ ]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=1, percentile_low=1.0, percentile_high=99.9, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
if zarr_path.exists():
  rm_tree(zarr_path)